# SESIÓN 1: DeepDream | La red que alucina: dos formas de ver el mundo

**Arqueología de modelos generativos · Taller MUTAR**

---

### Antes de empezar (2 minutos)

1. **Haz tu propia copia:** `Archivo → Guardar una copia en Drive`. Así tus cambios no se pierden.
2. **Activa la GPU:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar`.
   Sin GPU el notebook funciona, pero de 10 a 20 veces más lento.
3. **Ejecuta las celdas en orden, de arriba hacia abajo** (botón ▶ a la izquierda de cada celda, o `Ctrl + Enter`).
4. Las celdas con **formulario** (campos y menús a la derecha) se modifican sin tocar el código: cambia el valor y vuelve a ejecutar esa celda.

**Qué te llevas al final:** imagen A (texturas), imagen B (formas), una versión en alta resolución y, si quieres, un video.

## 0. Preparar el entorno

**¿Qué hace esta celda?**
Importa las bibliotecas necesarias para que la red neuronal pueda "soñar" sobre tu imagen y revisa si la GPU está activa.

**TensorFlow** es el framework de Google que usaremos, y **PIL/matplotlib** nos permiten ver y guardar los resultados.
Todo lo que generes se guarda en la carpeta `resultados` (panel de archivos 📁 a la izquierda).

In [ ]:
#@title 0. Preparar el entorno (ejecutar primero)
import os, io, time, glob, shutil, subprocess
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import requests
from IPython.display import display, Video
from tqdm.auto import tqdm
from google.colab import files, drive

print("TensorFlow versión:", tf.__version__)

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU activada. DeepDream está despierto.")
else:
    print("⚠️  NO hay GPU activa. El notebook funcionará, pero MUY lento.")
    print("    Ve a: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar")
    print("    y vuelve a ejecutar esta celda.")

# Carpeta donde se guardan todos los resultados
CARPETA = "/content/resultados"
os.makedirs(CARPETA, exist_ok=True)
print("📁 Los resultados se guardarán en:", CARPETA)

## ¿Qué es DeepDream?

**BREVE EXPLICACIÓN CONCEPTUAL**

DeepDream es una técnica publicada por Google en 2015 (Alexander Mordvintsev, Chris Olah y Mike Tyka, en el texto *"Inceptionism: Going Deeper into Neural Networks"*). No es una inteligencia artificial que "genera" imágenes desde cero (como hacen las GANs o los modelos de difusión). Es algo más extraño: es una red neuronal entrenada para **RECONOCER** objetos que, en vez de decirnos "esto es un perro", la obligamos a **MOSTRAR** lo que "ve".

La red que usamos es **InceptionV3**, entrenada con ImageNet: más de un millón de fotos etiquetadas en **1000 categorías**. Unas 120 de esas categorías son razas de perro, y por eso los perros aparecen por todas partes en los sueños de la red.

> Nota arqueológica: el DeepDream original de 2015 usaba **GoogLeNet** (Inception v1). InceptionV3 es su descendiente directo y es el que usa el tutorial oficial de TensorFlow. Las alucinaciones son parecidas, aunque no idénticas.

**¿Cómo funciona?**

1. Le damos una foto.
2. Le preguntamos a una capa específica de la red: "¿Qué activaciones (qué neuronas) se encienden al ver esta imagen?"
3. Calculamos el gradiente: ¿cómo debería cambiar la imagen para que ESAS neuronas se enciendan MÁS?
4. Modificamos la imagen en esa dirección.
5. Repetimos decenas o cientos de veces.

Es el proceso de entrenamiento **al revés**: al entrenar se cambia la red para que se ajuste a la imagen; aquí se cambia la imagen para que se ajuste a la red. La imagen se transforma gradualmente, amplificando los patrones que la red ya estaba buscando. Es como si la red "alucinara" sobre tu foto.

**CAPAS BAJAS vs CAPAS ALTAS: la clave de esta sesión**

Las redes neuronales profundas tienen muchas capas (como pisos de un edificio). Cada piso "ve" cosas diferentes:

* **CAPAS BAJAS** (`mixed0`, `mixed1`, `mixed2`)
  → Ven texturas, bordes, colores, formas simples.
  → El resultado es psicodélico, texturizado, abstracto.
  → Como ver el mundo a través de un caleidoscopio.

* **CAPAS MEDIAS** (`mixed3` a `mixed6`)
  → Ven partes: ojos, hocicos, plumas, ventanas, espirales.
  → Aquí empiezan a aparecer las "criaturas".

* **CAPAS ALTAS** (`mixed7` a `mixed10`)
  → Combinan esas partes en objetos más complejos y abstractos.
  → El resultado es más denso, figurativo y a veces caótico.

En esta sesión haremos AMBAS versiones con tu misma foto.

![Dogception](https://www.tensorflow.org/tutorials/generative/images/dogception.png)

# Instrucciones paso a paso

## 1. Cargar tu imagen

En el formulario de la derecha elige **de dónde viene tu imagen** y ejecuta la celda:

* **subir mi imagen**: aparece un botón para elegir un archivo de tu computadora.
* **URL de internet**: pega el enlace directo a una imagen (debe terminar en .jpg, .png, etc.). Algunos sitios bloquean la descarga; si falla, descárgala y súbela.
* **Google Drive**: pega la ruta de tu imagen. Para obtenerla: panel 📁 → `drive` → busca tu foto → clic derecho → **Copiar ruta**.
* **imagen de ejemplo (perro)**: para probar rápido.

Sirven fotos de celular (también se corrige la rotación), PNG con transparencia e imágenes en blanco y negro.

In [ ]:
#@title 1.1 Cargar imagen
FUENTE = "subir mi imagen" #@param ["subir mi imagen", "URL de internet", "Google Drive", "imagen de ejemplo (perro)"]
IMAGEN_URL = "" #@param {type:"string"}
RUTA_DRIVE = "/content/drive/MyDrive/tu-carpeta/tu-foto.jpg" #@param {type:"string"}
#@markdown Tamaño de trabajo (lado más largo, en píxeles). 512 es rápido; más grande = más detalle y más tiempo.
TAMANO_TRABAJO = 512 #@param [384, 512, 768, 1024] {type:"raw"}

EJEMPLO_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/YellowLabradorLooking_new.jpg"

def cargar_imagen(fuente):
    # Abre una imagen desde URL o archivo y la deja lista: RGB y bien rotada.
    if fuente.startswith("http"):
        r = requests.get(fuente, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content))
    else:
        if fuente.lower().endswith((".heic", ".heif")):          # fotos de iPhone
            subprocess.run("pip -q install pillow-heif", shell=True)
            import pillow_heif; pillow_heif.register_heif_opener()
        img = Image.open(fuente)
    img = ImageOps.exif_transpose(img)   # corrige fotos de celular que salen giradas
    return img.convert("RGB")            # quita transparencia / convierte blanco y negro a color

def ajustar_tamano(img, lado_max):
    # Escala la imagen para que su lado más largo mida 'lado_max' píxeles.
    escala = lado_max / max(img.size)
    nuevo = (max(1, round(img.width * escala)), max(1, round(img.height * escala)))
    return img.resize(nuevo, Image.LANCZOS)

def mostrar(img, titulo=""):
    plt.figure(figsize=(10, 10))
    plt.imshow(np.asarray(img))
    plt.title(titulo, fontsize=14, pad=10)
    plt.axis("off")
    plt.show()

def guardar(img, nombre):
    ruta = os.path.join(CARPETA, nombre)
    Image.fromarray(np.asarray(img)).save(ruta)
    print(f"💾 Guardado: {ruta}")
    return ruta

if FUENTE == "subir mi imagen":
    print("📤 Sube tu imagen desde tu computadora:")
    subidas = files.upload()
    if not subidas:
        raise SystemExit("⚠️ No subiste ninguna imagen. Vuelve a ejecutar la celda.")
    image_source = list(subidas.keys())[0]
elif FUENTE == "URL de internet":
    if not IMAGEN_URL.startswith("http"):
        raise SystemExit("⚠️ Pega una URL que empiece con http en el campo IMAGEN_URL.")
    image_source = IMAGEN_URL
elif FUENTE == "Google Drive":
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise SystemExit("⚠️ No se pudo conectar Drive. Revisa que el navegador permita ventanas emergentes "
                         "y que uses la misma cuenta de Google que en Colab. O usa 'subir mi imagen'.\n" + str(e))
    if not os.path.exists(RUTA_DRIVE):
        raise SystemExit(f"⚠️ No encuentro el archivo: {RUTA_DRIVE}\nUsa clic derecho → 'Copiar ruta' en el panel de archivos.")
    image_source = RUTA_DRIVE
else:
    image_source = EJEMPLO_URL

try:
    imagen_completa = cargar_imagen(image_source)      # guardamos la imagen a resolución completa
except Exception as e:
    raise SystemExit(f"⚠️ No se pudo abrir la imagen ({e}). Prueba con otra imagen u otra opción.")

original_img = np.array(ajustar_tamano(imagen_completa, TAMANO_TRABAJO))
mostrar(original_img, "Tu imagen original")
print(f"📎 Fuente: {image_source}")
print(f"📐 Resolución original: {imagen_completa.width} x {imagen_completa.height} px")
print(f"📐 Tamaño de trabajo:   {original_img.shape[1]} x {original_img.shape[0]} px")

## 2. Preparamos el modelo InceptionV3
![inception](https://www.researchgate.net/publication/364173580/figure/fig3/AS:11431281088122319@1664977637023/Architecture-of-Inception-v3.png)

**¿Qué es InceptionV3?**

Es una red neuronal entrenada por Google (2015) para clasificar imágenes en 1000 categorías. Ha visto más de un millón de fotos. No la entrenamos nosotros: la descargamos ya entrenada.

`include_top=False`: no queremos la parte final que dice "es un perro". Solo queremos las capas intermedias, donde la red "ve" patrones.

-----

InceptionV3 está organizada en **11 bloques "Inception"**, llamados **mixed0, mixed1, ..., mixed10**. Cada bloque tiene varias convoluciones en paralelo (de distintos tamaños) y cada uno ve el mundo a una "profundidad" diferente.

In [ ]:
#@title 2. Descargar InceptionV3 (ya entrenada)
print("Descargando InceptionV3 (ya entrenado)...")
base_model = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")
CAPAS_VALIDAS = [f"mixed{i}" for i in range(11)]
print("✅ Modelo cargado. Capas disponibles:", ", ".join(CAPAS_VALIDAS))

## 3. El motor de DeepDream

**¿Qué hace esta celda?** Define el "motor" que hace soñar a la red. No hace falta entender cada línea. La idea es:

* `calc_loss`: mide cuánto se "encienden" las capas elegidas.
* `paso`: empuja la imagen un poquito en la dirección que hace que se enciendan MÁS (*gradient ascent*: subimos, no bajamos).
* **Mosaico:** para imágenes grandes, la red mira la imagen por partes (baldosas de 512 px) y cada vez las desplaza al azar para que no se noten las costuras. Gracias a esto podemos soñar imágenes de 4K y hacer video.
* **Octavas:** soñar primero en pequeño y luego en grande (ver sección 4).

In [ ]:
#@title 3. Motor de DeepDream (solo ejecutar)
def calc_loss(img, model):
    activaciones = model(tf.expand_dims(img, axis=0))
    if not isinstance(activaciones, (list, tuple)):
        activaciones = [activaciones]
    return tf.add_n([tf.reduce_mean(a) for a in activaciones])

def _inicios(n, tile):
    # Posiciones de las baldosas: cubren toda la imagen y la última baldosa nunca es diminuta.
    ultimo = tf.maximum(n - tile, 0)
    return tf.concat([tf.range(0, ultimo, tile), [ultimo]], axis=0)

class DeepDream(tf.Module):
    # La magia de DeepDream en una caja (versión en mosaico, sirve para cualquier tamaño).
    def __init__(self, model, tile_size=512):
        self.model = model
        self.tile_size = tile_size

    @tf.function(input_signature=(tf.TensorSpec(shape=[None, None, 3], dtype=tf.float32),
                                  tf.TensorSpec(shape=[], dtype=tf.float32)))
    def paso(self, img, step_size):
        tile = self.tile_size
        desplazamiento = tf.random.uniform([2], -tile // 2, tile // 2, dtype=tf.int32)
        img_movida = tf.roll(img, shift=desplazamiento, axis=[0, 1])
        gradientes = tf.zeros_like(img_movida)
        perdida = tf.constant(0.0)
        for y in _inicios(tf.shape(img_movida)[0], tile):
            for x in _inicios(tf.shape(img_movida)[1], tile):
                with tf.GradientTape() as tape:
                    tape.watch(img_movida)
                    l = calc_loss(img_movida[y:y + tile, x:x + tile], self.model)
                gradientes = gradientes + tape.gradient(l, img_movida)
                perdida = perdida + l
        gradientes = tf.roll(gradientes, shift=-desplazamiento, axis=[0, 1])
        gradientes = gradientes / (tf.math.reduce_std(gradientes) + 1e-8)
        img = tf.clip_by_value(img + gradientes * step_size, -1.0, 1.0)   # Gradient ASCENT
        return img, perdida

def leer_capas(texto):
    capas = [c.strip() for c in texto.split(",") if c.strip()]
    malas = [c for c in capas if c not in CAPAS_VALIDAS]
    if not capas or malas:
        raise SystemExit(f"⚠️ Capas no válidas: {malas or texto!r}. Usa nombres entre mixed0 y mixed10, separados por comas.")
    return capas

def setup_dream_model(layer_names):
    # Crea un DeepDream que "escucha" las capas que elijamos.
    if isinstance(layer_names, str):
        layer_names = leer_capas(layer_names)
    salidas = [base_model.get_layer(nombre).output for nombre in layer_names]
    return DeepDream(tf.keras.Model(inputs=base_model.input, outputs=salidas))

def a_red(img_uint8):
    # Imagen normal (0-255) → formato de la red (-1 a 1)
    return tf.convert_to_tensor(tf.keras.applications.inception_v3.preprocess_input(np.array(img_uint8, dtype=np.float32)))

def a_imagen(img):
    # Formato de la red (-1 a 1) → imagen visible (0-255)
    return np.array(tf.cast(tf.round(255.0 * (img + 1.0) / 2.0), tf.uint8))

def sonar(img, deepdream, pasos, step_size, octavas=(0,), escala_octava=1.3, mostrar_progreso=False):
    # Sueña sobre una imagen ya en formato de la red. Devuelve el mismo formato y tamaño.
    base = tf.cast(tf.shape(img)[:2], tf.float32)
    step_size = tf.constant(step_size, tf.float32)
    for n in octavas:
        nuevo = tf.maximum(tf.cast(base * (escala_octava ** n), tf.int32), 100)
        img = tf.image.resize(img, nuevo)
        for i in range(pasos):
            img, perdida = deepdream.paso(img, step_size)
            if mostrar_progreso and len(octavas) == 1 and ((i + 1) % 25 == 0 or i + 1 == pasos):
                print(f"  Paso {i + 1}/{pasos} | intensidad del sueño: {float(perdida):.2f}")
        if mostrar_progreso and len(octavas) > 1:
            print(f"  Octava {n:+d} | tamaño {int(nuevo[1])} x {int(nuevo[0])} px | {pasos} pasos | intensidad: {float(perdida):.2f}")
    return tf.image.resize(img, tf.cast(base, tf.int32))

def run_deep_dream(img, deepdream, steps=100, step_size=0.01, octavas=(0,), escala_octava=1.3):
    # Versión simple: entra una imagen normal y sale una imagen normal.
    inicio = time.time()
    resultado = sonar(a_red(img), deepdream, steps, step_size, octavas, escala_octava, mostrar_progreso=True)
    print(f"⏱️ {time.time() - inicio:.1f} segundos")
    return a_imagen(resultado)

print("✅ Motor listo.")

## 3.1 Variante A: Texturas (capas bajas)

Usaremos las capas **mixed0** y **mixed1**. Estas son las primeras capas de la red, justo después de que la imagen entra. Aquí la red solo ha aprendido a detectar:

* Bordes y líneas
* Colores y contrastes
* Texturas simples
* Formas geométricas básicas

El resultado es orgánico, abstracto, como ver el mundo a través de un prisma o un caleidoscopio. No verás perros ni ojos: verás patrones que se repiten y se amplifican.

Analogía: es como si la red estuviera en estado de "hipnosis sensorial", fascinada por los colores y las texturas.

In [ ]:
#@title 3.1 Variante A: Texturas
CAPAS_A = "mixed0, mixed1" #@param {type:"string"}
PASOS_A = 100 #@param {type:"slider", min:10, max:400, step:10}
STEP_SIZE_A = 0.01 #@param {type:"number"}

print("VARIANTE A: Texturas y patrones (capas bajas)")
print("La red está 'soñando' con texturas...")
dream_texturas = setup_dream_model(CAPAS_A)
resultado_texturas = run_deep_dream(original_img, dream_texturas, steps=PASOS_A, step_size=STEP_SIZE_A)

mostrar(resultado_texturas, f"VARIANTE A: Texturas ({CAPAS_A}, {PASOS_A} pasos)")
guardar(resultado_texturas, "deepdream_A_texturas.png")

## 3.2 Variante B: Formas y objetos (capas medias y altas)

Usaremos las capas **mixed4**, **mixed5** y **mixed6**, más profundas en la red, y **más pasos** que en la variante A. Aquí la red ha aprendido a reconocer:

* Ojos, hocicos, plumas
* Caras de animales
* Edificios, arcos, objetos
* Formas complejas y reconocibles

El resultado es más figurativo. Verás ojos emergiendo de las texturas, perros escondidos en las nubes, caras en las paredes. Es como si la red estuviera OBSESIONADA con encontrar cosas donde no las hay.

Analogía: es como la pareidolia humana (ver caras en las nubes), pero llevada al extremo por una máquina que ha visto más de un millón de fotos.

**Prueba también:** `mixed7, mixed8` o `mixed9, mixed10` para ir aún más profundo. A veces el resultado es más caótico que figurativo: esa también es una forma de ver.

In [ ]:
#@title 3.2 Variante B: Formas
CAPAS_B = "mixed4, mixed5, mixed6" #@param {type:"string"}
PASOS_B = 150 #@param {type:"slider", min:10, max:400, step:10}
STEP_SIZE_B = 0.01 #@param {type:"number"}

print("VARIANTE B: Formas y objetos (capas profundas)")
print("La red está 'soñando' con objetos y caras...")
dream_formas = setup_dream_model(CAPAS_B)
resultado_formas = run_deep_dream(original_img, dream_formas, steps=PASOS_B, step_size=STEP_SIZE_B)

mostrar(resultado_formas, f"VARIANTE B: Formas ({CAPAS_B}, {PASOS_B} pasos)")
guardar(resultado_formas, "deepdream_B_formas.png")

## 3.3 Comparación lado a lado

In [ ]:
#@title 3.3 Comparación
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, titulo in zip(axes,
                           [original_img, resultado_texturas, resultado_formas],
                           ["Original", f"A · Texturas ({CAPAS_A})", f"B · Formas ({CAPAS_B})"]):
    ax.imshow(img); ax.set_title(titulo, fontsize=14); ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA, "comparacion_original_A_B.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Izquierda: tu foto original | Centro: la red soñando TEXTURAS | Derecha: la red soñando FORMAS")

## 4. OCTAVAS: escalas de la alucinación

¿QUÉ SON LAS OCTAVAS?

Hasta ahora procesamos la imagen a un solo tamaño. Las octavas nos permiten aplicar DeepDream a diferentes escalas, como si usáramos lentes de aumento de distintos poderes:

* **Octava negativa (-2, -1):** imagen más pequeña → la red ve patrones GRANDES, estructuras de lejos.
* **Octava 0:** tamaño de trabajo → la red ve detalles normales.
* **Octava positiva (1, 2):** imagen más grande → la red ve texturas finas, detalles microscópicos.

El truco: empezamos pequeño, la red "sueña" patrones grandes; luego agrandamos y la red añade detalles sobre esos patrones.

Es como pintar un mural: primero el boceto grande, luego los detalles.

**Para experimentar:** cambia los valores del formulario y vuelve a ejecutar **solo esta celda**. No hace falta correr todo el notebook otra vez. Cada resultado se guarda con un nombre distinto.

IDEAS PARA JUGAR:

1. **Más escalas (más denso):** OCTAVA_MIN = -3, OCTAVA_MAX = 3
2. **Saltos más bruscos entre escalas:** ESCALA_OCTAVA = 1.5
3. **Más intenso:** PASOS_POR_OCTAVA = 100
4. **Formas en octavas:** CAPAS_OCTAVAS = `mixed7, mixed8, mixed9`
5. **Capas intermedias (como el tutorial oficial):** CAPAS_OCTAVAS = `mixed3, mixed5`
6. **Partir de tu variante A o B** en vez de la foto original (PARTIR_DE)

In [ ]:
#@title 4. DeepDream con octavas
ESCALA_OCTAVA = 1.15 #@param {type:"slider", min:1.05, max:1.6, step:0.05}
OCTAVA_MIN = -1 #@param {type:"slider", min:-5, max:0, step:1}
OCTAVA_MAX = 1 #@param {type:"slider", min:0, max:4, step:1}
PASOS_POR_OCTAVA = 30 #@param {type:"slider", min:5, max:150, step:5}
STEP_SIZE_OCTAVAS = 0.01 #@param {type:"number"}
CAPAS_OCTAVAS = "mixed0, mixed1" #@param {type:"string"}
PARTIR_DE = "foto original" #@param ["foto original", "variante A", "variante B"]

punto_de_partida = {"foto original": original_img,
                    "variante A": resultado_texturas if "resultado_texturas" in globals() else original_img,
                    "variante B": resultado_formas if "resultado_formas" in globals() else original_img}[PARTIR_DE]
octavas = list(range(OCTAVA_MIN, OCTAVA_MAX + 1))
print(f"Iniciando DeepDream con {len(octavas)} octavas {octavas}, escala {ESCALA_OCTAVA}, capas {CAPAS_OCTAVAS}")

img_octavas = run_deep_dream(punto_de_partida, setup_dream_model(CAPAS_OCTAVAS),
                             steps=PASOS_POR_OCTAVA, step_size=STEP_SIZE_OCTAVAS,
                             octavas=octavas, escala_octava=ESCALA_OCTAVA)

mostrar(img_octavas, f"Octavas {OCTAVA_MIN}..{OCTAVA_MAX} · escala {ESCALA_OCTAVA} · {CAPAS_OCTAVAS}")
nombre = f"deepdream_octavas_{CAPAS_OCTAVAS.replace(', ', '-').replace(',', '-')}_{time.strftime('%H%M%S')}.png"
guardar(img_octavas, nombre)

## 5. Imagen en alta resolución (para imprimir o proyectar)

Hasta ahora trabajamos a 512 px, suficiente para ver en pantalla. Para una muestra necesitamos más: **2048 px** para proyectar en Full HD, **3000–4096 px** para imprimir.

Esta celda vuelve a tu foto original **a resolución completa** (si tu foto es más pequeña, la agranda) y sueña con octavas: empieza pequeño (estructuras grandes) y termina en el tamaño final (detalle fino). Gracias al mosaico, la memoria de la GPU alcanza incluso para 4K.

Tiempos aproximados en GPU T4: 2048 px ≈ 1–2 min · 4096 px ≈ 4–8 min.

In [ ]:
#@title 5. Alta resolución
LADO_MAYOR = 2048 #@param [1024, 1536, 2048, 3072, 4096] {type:"raw"}
CAPAS_HR = "mixed3, mixed5" #@param {type:"string"}
NUM_OCTAVAS = 4 #@param {type:"slider", min:1, max:6, step:1}
ESCALA_OCTAVA_HR = 1.4 #@param {type:"slider", min:1.1, max:1.8, step:0.1}
PASOS_POR_OCTAVA_HR = 30 #@param {type:"slider", min:5, max:100, step:5}
STEP_SIZE_HR = 0.01 #@param {type:"number"}

img_grande = np.array(ajustar_tamano(imagen_completa, LADO_MAYOR))
octavas_hr = list(range(-(NUM_OCTAVAS - 1), 1))     # termina siempre en el tamaño final
print(f"Soñando a {img_grande.shape[1]} x {img_grande.shape[0]} px con octavas {octavas_hr}...")

resultado_hr = run_deep_dream(img_grande, setup_dream_model(CAPAS_HR),
                              steps=PASOS_POR_OCTAVA_HR, step_size=STEP_SIZE_HR,
                              octavas=octavas_hr, escala_octava=ESCALA_OCTAVA_HR)
mostrar(resultado_hr, f"Alta resolución {resultado_hr.shape[1]} x {resultado_hr.shape[0]} px")
guardar(resultado_hr, f"deepdream_HR_{LADO_MAYOR}px_{time.strftime('%H%M%S')}.png")

## 6. Video: la alucinación en movimiento

Un video es una secuencia de imágenes (*frames*). Duración = número de frames ÷ frames por segundo (FPS). **10 segundos a 24 FPS = 240 frames.**

Hay dos videos posibles:

* **6.2 "La alucinación crece":** se ve cómo tu foto se va transformando, del original al sueño. Rápido de generar.
* **6.3 "Zoom infinito":** la clásica caída hacia dentro de la alucinación. Cada frame se acerca un poquito y la red vuelve a soñar sobre lo que aparece. Más lento, pero puede durar lo que quieras.

**Videos largos y grandes, cómo funciona:**

* Cada frame se guarda como imagen en una carpeta, así que si Colab se desconecta no pierdes lo hecho.
* Con **CONTINUAR = ✔** el zoom sigue desde el último frame y suma más segundos. Puedes renderizar por partes.
* Para no perder nada, ejecuta antes la celda **6.0** y guarda directamente en tu Drive.
* Los frames también te sirven para la **Sesión 4 (FILM)**: FILM puede inventar frames intermedios y hacer el video más fluido o más lento, sin volver a soñar.

| Resolución | Lado mayor | Zoom de 20 s a 24 FPS (T4, aprox.) |
|---|---|---|
| 480p | 854 | 2–4 min |
| 720p | 1280 | 4–8 min |
| 1080p | 1920 | 10–20 min |
| 4K | 3840 | 40–80 min (mejor por partes) |

La celda imprime una estimación de tiempo apenas empieza. Si es demasiado, detén la celda (■) y baja la resolución o la duración.

In [ ]:
#@title 6.0 (Opcional, recomendado para videos largos) Guardar todo en Google Drive
CARPETA_DRIVE = "/content/drive/MyDrive/Taller_MUTAR/sesion1" #@param {type:"string"}
try:
    drive.mount("/content/drive")
    os.makedirs(CARPETA_DRIVE, exist_ok=True)
    for f in glob.glob(os.path.join(CARPETA, "*.png")):        # copia lo que ya hiciste
        shutil.copy(f, CARPETA_DRIVE)
    CARPETA = CARPETA_DRIVE
    print("✅ A partir de ahora todo se guarda en tu Drive:", CARPETA)
except Exception as e:
    print("⚠️ No se pudo conectar Drive (permite ventanas emergentes y usa la misma cuenta). Se sigue guardando en", CARPETA)
    print(e)

In [ ]:
#@title 6.1 Formato del video (ejecutar antes de 6.2 / 6.3)
LADO_MAYOR_VIDEO = 1280 #@param [854, 1280, 1920, 2560, 3840] {type:"raw"}
FORMATO = "16:9 horizontal" #@param ["original", "16:9 horizontal", "9:16 vertical", "1:1 cuadrado", "4:5 retrato"]
FPS = 24 #@param [12, 24, 25, 30] {type:"raw"}
#@markdown CALIDAD: 18 = muy alta (archivo grande), 23 = buena, 28 = liviana (para WhatsApp)
CALIDAD = 18 #@param {type:"slider", min:14, max:30, step:1}

PROPORCIONES = {"16:9 horizontal": 16 / 9, "9:16 vertical": 9 / 16, "1:1 cuadrado": 1.0, "4:5 retrato": 4 / 5}

def preparar_lienzo(img_pil, lado_mayor, formato):
    # Recorta al centro con la proporción elegida y escala. Las medidas quedan pares (lo exige el formato mp4).
    if formato in PROPORCIONES:
        p = PROPORCIONES[formato]
        w, h = img_pil.size
        if w / h > p:
            nw = round(h * p); img_pil = img_pil.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
        else:
            nh = round(w / p); img_pil = img_pil.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    img_pil = ajustar_tamano(img_pil, lado_mayor)
    w, h = img_pil.size
    return np.array(img_pil.crop((0, 0, w - w % 2, h - h % 2)))

def ffmpeg():
    return shutil.which("ffmpeg") or __import__("imageio_ffmpeg").get_ffmpeg_exe()

def carpeta_frames(nombre, continuar):
    carpeta = os.path.join(CARPETA, f"frames_{nombre}")
    if os.path.exists(carpeta) and not continuar:
        shutil.rmtree(carpeta)
    os.makedirs(carpeta, exist_ok=True)
    return carpeta, sorted(glob.glob(os.path.join(carpeta, "*.jpg")))

def guardar_frame(img, carpeta, i):
    Image.fromarray(a_imagen(img)).save(os.path.join(carpeta, f"{i:06d}.jpg"), quality=95)

def montar_video(nombre, fps=None, ida_y_vuelta=False, calidad=None, gif=False, mostrar_video=True):
    # Une los frames de la carpeta en un .mp4 (y opcionalmente un .gif liviano).
    fps, calidad = fps or FPS, calidad or CALIDAD
    frames = sorted(glob.glob(os.path.join(CARPETA, f"frames_{nombre}", "*.jpg")))
    if not frames:
        raise SystemExit(f"⚠️ No hay frames para '{nombre}'.")
    if ida_y_vuelta:
        frames = frames + frames[-2:0:-1]
    tmp = "/content/_montaje"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for i, f in enumerate(frames):
        os.symlink(f, os.path.join(tmp, f"{i:06d}.jpg"))
    salida = os.path.join(CARPETA, f"{nombre}.mp4")
    subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-framerate", str(fps), "-i", f"{tmp}/%06d.jpg",
                    "-c:v", "libx264", "-preset", "medium", "-crf", str(calidad),
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", salida], check=True)
    mb = os.path.getsize(salida) / 1e6
    print(f"🎬 Video: {salida} | {len(frames)} frames | {len(frames) / fps:.1f} s | {mb:.1f} MB")
    if gif:
        salida_gif = os.path.join(CARPETA, f"{nombre}.gif")
        subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-i", salida, "-vf",
                        "fps=12,scale=480:-2:flags=lanczos,split[a][b];[a]palettegen[p];[b][p]paletteuse",
                        salida_gif], check=True)
        print(f"🎞️ GIF:   {salida_gif} | {os.path.getsize(salida_gif) / 1e6:.1f} MB")
    if mostrar_video and mb < 40:
        display(Video(salida, embed=True, width=640))
    elif mostrar_video:
        print("(Video muy pesado para mostrarlo aquí: descárgalo en la sección 7.)")
    return salida

def estimar(inicio, hechos, total):
    seg = (time.time() - inicio) / hechos * (total - hechos)
    print(f"⏱️ Tiempo estimado restante: {seg / 60:.1f} min ({total} frames en total)")

lienzo_video = preparar_lienzo(imagen_completa, LADO_MAYOR_VIDEO, FORMATO)
mostrar(lienzo_video, f"Lienzo del video: {lienzo_video.shape[1]} x {lienzo_video.shape[0]} px")

### 6.2 Video "La alucinación crece"

La foto original se transforma de a poco en el sueño. **INTENSIDAD** define cuán "soñado" queda el último frame (equivale a los pasos de las variantes A y B). La **DURACIÓN** solo cambia la velocidad: el final es el mismo.

Con **IDA_Y_VUELTA** el video vuelve hacia la foto original (el doble de largo, gratis).

In [ ]:
#@title 6.2 Video: la alucinación crece
NOMBRE_VIDEO_A = "crece" #@param {type:"string"}
CAPAS_VIDEO_A = "mixed4, mixed5" #@param {type:"string"}
DURACION_SEG_A = 10 #@param {type:"slider", min:2, max:120, step:1}
INTENSIDAD = 150 #@param {type:"slider", min:20, max:400, step:10}
IDA_Y_VUELTA = True #@param {type:"boolean"}
HACER_GIF_A = False #@param {type:"boolean"}

dd_video_a = setup_dream_model(CAPAS_VIDEO_A)
n_frames = int(DURACION_SEG_A * FPS)
paso_por_frame = INTENSIDAD * 0.01 / n_frames      # repartimos la intensidad entre todos los frames
carpeta, _ = carpeta_frames(NOMBRE_VIDEO_A, continuar=False)

img = a_red(lienzo_video)
inicio = time.time()
for i in tqdm(range(n_frames), desc="Soñando frames"):
    guardar_frame(img, carpeta, i)
    img = sonar(img, dd_video_a, pasos=1, step_size=paso_por_frame)
    if i == 4:
        estimar(inicio, 5, n_frames)

montar_video(NOMBRE_VIDEO_A, ida_y_vuelta=IDA_Y_VUELTA, gif=HACER_GIF_A)

### 6.3 Video "Zoom infinito"

Cada frame: la red sueña unos pocos pasos → acercamos la cámara un poquito → repetimos. Lo que aparece en el centro se agranda y la red le sigue agregando alucinaciones.

* **ZOOM_POR_SEGUNDO:** cuánto se acerca por segundo (0.3 = 30%). Más alto = caída más rápida.
* **PASOS_POR_FRAME:** cuánto sueña en cada frame. Más = más denso (y más lento).
* **CENTRO_X / CENTRO_Y:** hacia qué punto cae la cámara (0.5, 0.5 = el centro).
* **GIRO_POR_SEGUNDO:** rotación en grados por segundo (0 = sin giro).
* **CONTINUAR:** ✔ para sumar más segundos al video anterior con el mismo nombre (renderizar por partes).

In [ ]:
#@title 6.3 Video: zoom infinito
NOMBRE_VIDEO_Z = "zoom" #@param {type:"string"}
CAPAS_ZOOM = "mixed3, mixed5" #@param {type:"string"}
DURACION_SEG_Z = 20 #@param {type:"slider", min:2, max:300, step:1}
ZOOM_POR_SEGUNDO = 0.3 #@param {type:"slider", min:0.05, max:1.5, step:0.05}
PASOS_POR_FRAME = 4 #@param {type:"slider", min:1, max:20, step:1}
STEP_SIZE_ZOOM = 0.01 #@param {type:"number"}
PASOS_INICIALES = 60 #@param {type:"slider", min:0, max:300, step:10}
CENTRO_X = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
CENTRO_Y = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
GIRO_POR_SEGUNDO = 0 #@param {type:"slider", min:-20, max:20, step:1}
CONTINUAR = False #@param {type:"boolean"}
HACER_GIF_Z = False #@param {type:"boolean"}

def mover_camara(img, factor, cx, cy, grados):
    # Acerca la imagen 'factor' veces hacia (cx, cy) y la gira 'grados'. Movimiento suave, sin saltos de píxel.
    h, w = img.shape[0], img.shape[1]
    mitad = 0.5 / factor
    cy = min(max(cy, mitad), 1 - mitad); cx = min(max(cx, mitad), 1 - mitad)
    caja = [[cy - mitad, cx - mitad, cy + mitad, cx + mitad]]
    img = tf.image.crop_and_resize(img[None], caja, [0], [h, w], method="bilinear")[0]
    if grados:
        t = np.deg2rad(grados); c, s = np.cos(t), np.sin(t)
        x0 = w / 2 - c * w / 2 + s * h / 2; y0 = h / 2 - s * w / 2 - c * h / 2
        transf = tf.constant([[c, -s, x0, s, c, y0, 0, 0]], tf.float32)
        img = tf.raw_ops.ImageProjectiveTransformV3(images=img[None], transforms=transf,
                                                     output_shape=[h, w], fill_value=0.0,
                                                     interpolation="BILINEAR", fill_mode="REFLECT")[0]
    return img

dd_zoom = setup_dream_model(CAPAS_ZOOM)
carpeta, existentes = carpeta_frames(NOMBRE_VIDEO_Z, CONTINUAR)
n_frames = int(DURACION_SEG_Z * FPS)
factor = (1 + ZOOM_POR_SEGUNDO) ** (1 / FPS)
giro = GIRO_POR_SEGUNDO / FPS

if CONTINUAR and existentes:
    img = a_red(np.array(Image.open(existentes[-1])))
    primero = len(existentes)
    print(f"▶️ Continuando desde el frame {primero} ({primero / FPS:.1f} s ya hechos)")
else:
    img = a_red(lienzo_video)
    primero = 0
    if PASOS_INICIALES:
        print("Sueño inicial...")
        img = sonar(img, dd_zoom, PASOS_INICIALES, STEP_SIZE_ZOOM, octavas=(-2, -1, 0), escala_octava=1.4)

inicio = time.time()
for i in tqdm(range(n_frames), desc="Zoom"):
    guardar_frame(img, carpeta, primero + i)
    img = mover_camara(img, factor, CENTRO_X, CENTRO_Y, giro)
    img = sonar(img, dd_zoom, PASOS_POR_FRAME, STEP_SIZE_ZOOM)
    if i == 4:
        estimar(inicio, 5, n_frames)

montar_video(NOMBRE_VIDEO_Z, gif=HACER_GIF_Z)

In [ ]:
#@title 6.4 (Opcional) Volver a montar un video con otros ajustes
#@markdown Sirve para cambiar FPS, calidad, hacer ida y vuelta o un GIF sin volver a soñar los frames.
#@markdown Ejemplo: los mismos frames a 12 FPS = video el doble de largo y más lento.
NOMBRE = "zoom" #@param {type:"string"}
FPS_MONTAJE = 24 #@param [8, 12, 15, 24, 25, 30] {type:"raw"}
IDA_Y_VUELTA_M = False #@param {type:"boolean"}
CALIDAD_M = 18 #@param {type:"slider", min:14, max:30, step:1}
HACER_GIF_M = True #@param {type:"boolean"}

montar_video(NOMBRE, fps=FPS_MONTAJE, ida_y_vuelta=IDA_Y_VUELTA_M, calidad=CALIDAD_M, gif=HACER_GIF_M)

## 7. Descargar tus resultados

Descarga un `.zip` con todo lo que generaste (imágenes, videos y GIFs). Los frames sueltos no se incluyen por defecto porque pesan mucho; si los quieres para la Sesión 4, marca INCLUIR_FRAMES o guárdalos en Drive (celda 6.0).

**Para la Sesión 4 (FILM) guarda:** `deepdream_A_texturas.png` y `deepdream_B_formas.png`. Tienen exactamente el mismo tamaño, así que FILM puede interpolar entre ellas.

In [ ]:
#@title 7. Descargar todo en un .zip
INCLUIR_FRAMES = False #@param {type:"boolean"}

zip_base = "/content/sesion1_deepdream"
tmp = "/content/_para_zip"
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(CARPETA, tmp, ignore=None if INCLUIR_FRAMES else shutil.ignore_patterns("frames_*"))
ruta_zip = shutil.make_archive(zip_base, "zip", tmp)
print(f"📦 {ruta_zip} ({os.path.getsize(ruta_zip) / 1e6:.1f} MB)")
if os.path.getsize(ruta_zip) > 500e6:
    print("⚠️ Es muy pesado para descargar desde el navegador: usa la celda 6.0 para guardarlo en Drive.")
files.download(ruta_zip)

---
### Para seguir explorando

* Tutorial oficial de TensorFlow: https://www.tensorflow.org/tutorials/generative/deepdream
* Texto original de Google (2015), *Inceptionism: Going Deeper into Neural Networks*: https://research.google/blog/inceptionism-going-deeper-into-neural-networks/
* *Feature Visualization* (Olah, Mordvintsev, Schubert, Distill 2017): https://distill.pub/2017/feature-visualization/
* Implementación en PyTorch (gordicaleksa): https://github.com/gordicaleksa/pytorch-deepdream